In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [2]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [3]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [5]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [6]:
rec = answers[0]
rec

{'question': 'I just found this course — is it too late to join now?',
 'answer_llm': 'Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [7]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
I just found this course — is it too late to join now?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [8]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: it says you can still join, but to receive a certificate you must submit the project while submissions are open. This is semantically equivalent.', score='good')

In [9]:
calc_price(usage)

{'input_cost': 0.00021825000000000002,
 'output_cost': 0.000252,
 'total_cost': 0.00047025}

In [11]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [12]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key message: it is not too late to join, but a certificate requires submitting the project while submissions are still open. This is semantically equivalent to the ground truth.', score='good')

In [13]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [15]:
answers

[{'question': 'I just found this course — is it too late to join now?',
  'answer_llm': 'Yes, you can still join now. If you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': "Can I still start the course if I'm coming in late?",
  'answer_llm': 'Yes — you can start whenever you want. The videos and GitHub materials are available, and you can follow the course workflow at your own pace.\n\nIf you want to receive a certificate, make sure you submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I enroll now, is there any chance to get a certificate?',
  'answer_llm': 'Yes —

In [16]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/560 [00:00<?, ?it/s]

In [17]:
results[10]

({'question': 'How do students join the Office Hours or live workshop sessions if the Zoom link isn’t public?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: it correctly says the Zoom link is only for instructors/presenters/TAs, and that students join via YouTube Live, can watch on the DataTalksClub YouTube channel, use the video URL posted in Telegram/Slack announcements, and submit questions via Slido. It also correctly warns not to post questions in chat. Semantically equivalent.'},
 ResponseUsage(input_tokens=452, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=100, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=552))

In [ ]:
# ({'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
#   'document': '489dd1c9d9',
#   'score': 'good',
#   'reasoning': "The AI answer matches the ground truth: it states students don't need the Zoom link, that participation is via YouTube Live, the URL is posted in the Telegram/Slack announcements channel, questions go through Slido, and the YouTube channel is available. It omits the caution about not posting questions in chat, but that is ancillary and doesn't change the core answer."},
#  ResponseUsage(input_tokens=433, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=90, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=523))

In [19]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [25]:
evaluations


[{'question': 'I just found this course — is it too late to join now?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the key meaning of the original: it says it is still possible to join, and that getting a certificate requires submitting the project before submissions close. This is semantically equivalent.'},
 {'question': "Can I still start the course if I'm coming in late?",
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the key point: late enrollment is allowed, course materials are available, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent to the ground truth.'},
 {'question': 'If I enroll now, is there any chance to get a certificate?',
  'document': '74eb249bbf',
  'score': 'good',
  'reasoning': 'The AI answer preserves the key point: a certificate is possible only if the project is submitted while submissions are still open.

In [26]:
usages

[ResponseUsage(input_tokens=291, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=55, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=346),
 ResponseUsage(input_tokens=314, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=56, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=370),
 ResponseUsage(input_tokens=316, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=73, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=389),
 ResponseUsage(input_tokens=371, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=91, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=462),
 ResponseUsage(input_tokens=329, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=93, output_tokens_det

In [27]:
calc_total_price(usages)

0.3935767499999999

In [28]:
df_eval = pd.DataFrame(evaluations)

In [29]:
df_eval.head()

,question,document,score,reasoning
0,I just found this course — is it too late to j...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still start the course if I'm coming in ...,74eb249bbf,good,The AI answer preserves the key point: late en...
2,"If I enroll now, is there any chance to get a ...",74eb249bbf,good,The AI answer preserves the key point: a certi...
3,What do I need to do to be eligible for the co...,74eb249bbf,bad,The ground truth only says that to receive a c...
4,Is the final project deadline the only thing t...,74eb249bbf,bad,The ground truth says yes: the final project d...


In [30]:
df_eval.score.value_counts()

score
good    537
bad      23
Name: count, dtype: int64

In [31]:
df_eval.score.value_counts(normalize=True)

score
good    0.958929
bad     0.041071
Name: proportion, dtype: float64

In [32]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
3,What do I need to do to be eligible for the co...,74eb249bbf,bad,The ground truth only says that to receive a c...
4,Is the final project deadline the only thing t...,74eb249bbf,bad,The ground truth says yes: the final project d...
16,"If I’m starting late, can I still join the cou...",04919992b3,bad,The AI answer correctly conveys that a student...
31,Is the capstone project the only thing I need ...,9f689c185f,bad,The AI answer directly contradicts the ground ...
33,"If homework is optional, does it still matter ...",9f689c185f,bad,The AI answer does not convey the ground truth...


In [33]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)